[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/labs/lab-07-cosine-tfidf.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Lab 7 — Cosine similarity and TF-IDF

**MSIS · AME 5053 · Week 7 · 3 hours**

Session 19 derived cosine similarity on four toy vectors and left one number on the
board: a document and the same document printed twice have a cosine of exactly 1.
This lab checks that in code, then builds the two things the lectures kept promising —
a **TF-IDF vectorizer** and a **search engine that ranks by cosine** — and points both
at 2,000 real film reviews.

Along the way three things happen that the lectures could not show you:

1. You finish an exercise the textbook leaves undone.
2. `scikit-learn` disagrees with the lecture's formula, and you find out exactly why.
3. You discover whether TF-IDF actually *helps*, which is not the question anyone
   asked in session 18.

**Covers L2.3** (cosine similarity and its significance in NLP) **and L2.4**
(word vectorization: TF-IDF).

---

## Part 0 — Setup

Same as lab 6: NLTK is on Colab already, its corpora are not, and they vanish when
Colab recycles the machine.

In [ ]:
!pip install -q nltk scikit-learn

import nltk

# No quiet=True: if the download fails we want to see it, not carry on with no data.
ok = nltk.download("movie_reviews")
print("movie_reviews downloaded:", ok)

import numpy as np
import sklearn
print("numpy", np.__version__, "· scikit-learn", sklearn.__version__)
print("Done.")

> **Save your own copy now:** File → Save a copy in Drive.

If `movie_reviews downloaded: False`, re-run the cell — almost always a network
hiccup. Parts 1–3 work without it; Parts 4–7 do not.

---

## Part 1 — Cosine from scratch, in NumPy

The syllabus says NumPy for this part, and it is right to: the formula is three lines,
and writing it yourself once is worth more than importing it a hundred times.

Here it is from session 19:

$$\cos(a, b) = \frac{a \cdot b}{|a|\,|b|} \qquad\text{where}\qquad |a| = \sqrt{\textstyle\sum_i a_i^2}$$

Two NumPy pieces do all the work:

- `a @ b` — the dot product, $\sum_i a_i b_i$
- `np.linalg.norm(a)` — the length $|a|$

Write the function.

In [ ]:
import numpy as np

def cosine(a, b):
    """Cosine similarity between two 1-D NumPy arrays."""
    # YOUR CODE HERE
    # dot product, divided by the product of the two lengths
    pass


def euclidean(a, b):
    """Straight-line distance between two 1-D NumPy arrays."""
    # YOUR CODE HERE
    pass

### Check it against the board

Session 19 opened on *Julius Caesar*'s raw column and the same play printed twice.
Euclidean distance called them completely different; cosine called them identical.

If your function is right, you get 62.434 and exactly 1.

In [ ]:
d  = np.array([7.0, 62, 1, 2])      # Julius Caesar: battle, good, fool, wit
d2 = 2 * d                          # the same play, printed twice

print("|d|  =", round(float(np.linalg.norm(d)), 4))
print("|2d| =", round(float(np.linalg.norm(d2)), 4))
print("euclidean(d, 2d) =", round(euclidean(d, d2), 4))
print("cosine(d, 2d)    =", cosine(d, d2))

assert np.isclose(np.linalg.norm(d), 62.4340, atol=1e-4)
assert np.isclose(euclidean(d, d2), 62.4340, atol=1e-4)
assert np.isclose(cosine(d, d2), 1.0)

# And it is not special to doubling. Any positive multiple:
for k in [0.5, 3, 1000]:
    assert np.isclose(cosine(d, k * d), 1.0)
print("\nScaling by 0.5, 3 and 1000 all give cosine 1.0 as well.")

Session 19's recall box, the smallest possible version of the same idea — small enough
that you were asked to do it in your head:

In [ ]:
a = np.array([3.0, 0])
b = np.array([30.0, 0])

print("euclidean =", euclidean(a, b))
print("cosine    =", cosine(a, b))

assert euclidean(a, b) == 27.0
assert np.isclose(cosine(a, b), 1.0)
print("\n27 and 1.0 — the two measurements disagreeing as loudly as they can.")

### The Shakespeare result

Session 19's payoff. Session 17's matrix, log-damped (`tf = 1 + log₁₀ count`) and with
**no idf**, gave every play a nearest neighbour in its own genre. Rebuild it and check
all six pairs.

In [ ]:
plays = ["As You Like It", "Twelfth Night", "Julius Caesar", "Henry V"]
genre = ["comedy", "comedy", "history", "history"]
words = ["battle", "good", "fool", "wit"]

# Session 17's raw counts: rows are words, columns are plays
raw = np.array([
    [1.0,   0,  7, 13],   # battle
    [114.0, 80, 62, 89],  # good
    [36.0,  58,  1,  4],  # fool
    [20.0,  15,  2,  3],  # wit
])

# log damping, exactly session 18's tf: 1 + log10(count), and 0 when the count is 0
logtf = np.where(raw > 0, 1 + np.log10(np.where(raw > 0, raw, 1)), 0.0)

print("log-tf matrix:")
print("           ", "  ".join(f"{p[:12]:>12}" for p in plays))
for w, row in zip(words, logtf):
    print(f"{w:>10} ", "  ".join(f"{v:12.3f}" for v in row))

In [ ]:
print("All six cosines, largest first:\n")

pairs = []
for i in range(4):
    for j in range(i + 1, 4):
        c = cosine(logtf[:, i], logtf[:, j])
        same = "same genre" if genre[i] == genre[j] else ""
        pairs.append((c, plays[i], plays[j], same))

for c, p1, p2, same in sorted(pairs, reverse=True):
    print(f"  {c:.4f}   {p1:<15} <-> {p2:<15} {same}")

# The two same-genre pairs are the top two. Session 19's claim, checked.
top_two = sorted(pairs, reverse=True)[:2]
assert all(same == "same genre" for _, _, _, same in top_two)

# and the individual numbers from the lesson
assert np.isclose(cosine(logtf[:, 0], logtf[:, 1]), 0.9753, atol=1e-4)   # AYLI <-> TN
assert np.isclose(cosine(logtf[:, 2], logtf[:, 3]), 0.9925, atol=1e-4)   # JC   <-> HV
assert np.isclose(cosine(logtf[:, 1], logtf[:, 2]), 0.8059, atol=1e-4)   # TN   <-> JC
print("\nThe two same-genre pairs are the two highest cosines. Session 19 checks out.")

### The same formula on *word* vectors

Everything above compared documents. Session 19 closed with SLP3's own example, which
compares **words** — each word represented by how often it appears near `pie`, `data`
and `computer`.

This is the idea sessions 21–22 are built on, so it is worth seeing the arithmetic is
identical.

In [ ]:
#                        pie   data  computer
cherry      = np.array([442.0,    8,     2])
digital     = np.array([  5.0, 1683,  1670])
information = np.array([  5.0, 3982,  3325])

print("cos(cherry, information)  =", round(cosine(cherry, information), 4))
print("cos(digital, information) =", round(cosine(digital, information), 4))

assert np.isclose(cosine(cherry, information), 0.018, atol=1e-3)
assert np.isclose(cosine(digital, information), 0.996, atol=1e-3)

print("\nlengths:", [round(float(np.linalg.norm(v)), 1)
                    for v in (cherry, digital, information)])
print("\n'information' has by far the longest vector — it is simply a commoner word.")
print("The raw dot product would have made every comparison with it look strong.")
print("Dividing by the lengths is what stops that.")

### One import, and we can stop writing it by hand

`sklearn.metrics.pairwise.cosine_similarity` does all pairs at once and works on the
sparse matrices we are about to build. Check it agrees with your function, then trust
it for the rest of the lab.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# cosine_similarity works on rows, our plays are columns, so transpose
S = cosine_similarity(logtf.T)

print("pairwise cosine matrix (4 plays):")
print(np.round(S, 4))

mine = np.array([[cosine(logtf[:, i], logtf[:, j]) for j in range(4)] for i in range(4)])
assert np.allclose(S, mine)
print("\nIdentical to the function you wrote. On to real data.")

---

## Part 2 — Finish the exercise the textbook leaves undone

SLP3 §11.1.3 works a tiny retrieval example: the query `sweet love` against four
nano-documents. It computes the tf-idf cosine for documents 1 and 2, prints 0.747 and
0.0779 — session 19 quoted both — and then says:

> *"computations for Documents 3 and 4 are also needed but are left as an exercise for
> the reader"*

So: reproduce the two the book gives, which tells you your code is right, and then
compute the two it doesn't.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = [
    "Sweet sweet nurse! Love?",
    "Sweet sorrow",
    "How sweet is love?",
    "Nurse!",
]
query = "sweet love"

cv = CountVectorizer()
X = cv.fit_transform(docs).toarray().astype(float)
vocab = cv.get_feature_names_out()
N = len(docs)
df = (X > 0).sum(axis=0)

print("vocabulary:", list(vocab))
print("\ncounts (rows = documents):")
print(X.astype(int))
print("\ndf:", dict(zip(vocab, df.tolist())))

Now build the tf-idf weights **using session 18's formula**, the one from the lecture:

$$\text{tf} = 1 + \log_{10}(\text{count}) \ \ (\text{and } 0 \text{ when the count is } 0) \qquad
\text{idf} = \log_{10}\!\left(\frac{N}{\text{df}}\right)$$

Write the function so it works on a whole matrix of counts at once.

In [ ]:
def tfidf_slp3(counts, df, N):
    """tf-idf per session 18: tf = 1 + log10(count), idf = log10(N/df)."""
    # YOUR CODE HERE
    # Careful with log10(0) — the tf is 0 when the count is 0, not -inf.
    # np.where(counts > 0, ..., 0.0) is the tidy way.
    pass

In [ ]:
scores_hand = cosine_similarity(q_hand, D_hand)[0]

print("cosine of the query 'sweet love' with each document:\n")
for i, (doc, s) in enumerate(zip(docs, scores_hand), start=1):
    book = {1: "  <- book says 0.747", 2: "  <- book says 0.0779"}.get(i, "  <- the book's exercise")
    print(f"  d{i}  {s:.4f}   {doc:<26}{book}")

# The two the book prints:
assert np.isclose(scores_hand[0], 0.747, atol=1e-3)
assert np.isclose(scores_hand[1], 0.0779, atol=1e-4)
print("\nBoth of the book's numbers reproduce. So the other two can be trusted.")

### Read the two you just computed

**d3 — `How sweet is love?` — 0.3575.** It contains both query words, same as d1, but
scores less than half of d1. Why? d1 has `sweet` *twice*, and d3 pads its vector with
`how` and `is`, which the query does not want and which pull its direction away.

**d4 — `Nurse!` — exactly 0.** Not nearly zero. The document and the query have no word
in common, so every product in the dot product has a zero in it. This is the 0 end of
session 19's range, and it is the common case in real search: most documents share
nothing with most queries.

The full ranking is **d1, d3, d2, d4** — which is the sensible order, and note the book
only ever told you where d1 and d2 sat.

---

## Part 3 — Meet `TfidfVectorizer`, and a disagreement

L2.4 is TF-IDF vectorization with scikit-learn. In practice that is one class, and it
replaces everything you just wrote:

```python
TfidfVectorizer().fit_transform(docs)
```

Run it on the same four documents and compare.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer()
D_sk = tv.fit_transform(docs)
q_sk = tv.transform([query])

scores_sk = cosine_similarity(q_sk, D_sk)[0]

print(f"{'':4} {'by hand (lecture)':>18} {'TfidfVectorizer':>18}")
for i in range(4):
    print(f"  d{i+1} {scores_hand[i]:18.4f} {scores_sk[i]:18.4f}")

# Verified output:
#         by hand (lecture)    TfidfVectorizer
#     d1             0.7469             0.8354
#     d2             0.0779             0.3385
#     d3             0.3575             0.5829
#     d4             0.0000             0.0000

**Every number is different.**

Before going on: in labs 5 and 6 a disagreement like this turned out to be one
settable option, and the two columns closed to exactly the same number. **This one does
not.** It narrows and it never closes, and that is not a bug in your code or in
sklearn. Do not go hunting for the keyword that fixes it — there isn't one.

What there is instead is a formula you can read. Start with the idf, which
`TfidfVectorizer` exposes.

In [ ]:
print(f"{'word':>8} {'df':>4} {'lecture log10(N/df)':>21} {'sklearn idf_':>14}")
for w, d_, hand_idf, sk_idf in zip(vocab, df, np.log10(N / df), tv.idf_):
    print(f"{w:>8} {d_:>4} {hand_idf:21.4f} {sk_idf:14.4f}")

print("\nNot a constant factor apart — look at 'sweet' (0.1249 vs 1.2231) against")
print("'how' (0.6021 vs 1.9163). Something structural is different.")

### What sklearn actually computes

Straight from the `TfidfTransformer` documentation:

> The formula that is used to compute the tf-idf for a term *t* of a document *d* is
> `tf-idf(t, d) = tf(t, d) * idf(t)`, and the idf is computed as
> `idf(t) = log[n / df(t)] + 1` (if `smooth_idf=False`) […] If `smooth_idf=True` (the
> default), the constant "1" is added to the numerator and denominator of the idf as if
> an extra document was seen containing every term in the collection exactly once:
> `idf(t) = log[(1 + n) / (1 + df(t))] + 1`.

So, three differences from the lecture:

| | lecture (SLP3) | sklearn default |
|---|---|---|
| tf | `1 + log₁₀(count)` | **raw count** |
| log base | 10 | **natural** |
| idf | `log(N/df)` | `log((1+N)/(1+df))` **+ 1** |
| after | nothing | **L2-normalise each row** |

The `+1` on the idf is the one that cannot be switched off, and sklearn explains why:

> The effect of adding "1" to the idf in the equation above is that terms with **zero
> idf**, i.e. terms that occur in all documents in a training set, **will not be
> entirely ignored**.

Read that against session 18. That lesson spent a whole hour on `good` collapsing to
*exactly* zero and taking its whole row with it. **sklearn considers that behaviour
undesirable and has deliberately engineered it away.** The library and the textbook
disagree about something real, and both of them are stating a defensible position.

Now prove you understand it: reproduce sklearn's matrix by hand.

In [ ]:
def tfidf_sklearn(counts, df, N):
    """Reproduce TfidfVectorizer's default output from raw counts.

    Three steps: idf with the smoothing and the +1, multiply by the raw counts,
    then L2-normalise each row (divide it by its own length).
    """
    # YOUR CODE HERE
    # np.log is the natural log. np.linalg.norm(..., axis=1, keepdims=True)
    # gives you one length per row.
    pass

### So which one is right?

Neither, and that is the answer. There is no single TF-IDF — there is a **family** of
weighting schemes, and IIR chapter 6 tabulates them under the name *SMART notation*.
The `TfidfTransformer` docstring names its own position in that table: tf is `n`
(natural) by default and `l` (logarithmic) with `sublinear_tf=True`; normalisation is
`c` (cosine) with `norm='l2'`.

Which raises the question that actually matters. The numbers differ — **does the answer
differ?** Retrieval does not consume the cosine values. It consumes the *order*.

In [ ]:
variants = {
    "lecture: 1+log10(count), log10(N/df)": scores_hand,
    "sklearn defaults": scores_sk,
    "sklearn, sublinear_tf + smooth_idf=False + norm=None": None,
}

tv2 = TfidfVectorizer(sublinear_tf=True, smooth_idf=False, norm=None)
D2 = tv2.fit_transform(docs)
variants["sklearn, sublinear_tf + smooth_idf=False + norm=None"] = \
    cosine_similarity(tv2.transform([query]), D2)[0]

for name, s in variants.items():
    order = [f"d{i+1}" for i in np.argsort(-s)]
    print(f"{np.round(s, 4)}   ranking: {' > '.join(order)}")
    print(f"    {name}\n")

orders = [tuple(np.argsort(-s)) for s in variants.values()]
assert len(set(orders)) == 1
print("Three different sets of numbers. One ranking, identical in all three.")

**That is the resolution.** The variants disagree about the magnitudes and agree about
the order, and a search engine only ever shows you the order.

It is worth being precise about how far that goes: this is four documents and a
two-word query, so the ranking agreeing is reassuring rather than proven. Variants
*can* reorder results on real collections — which is exactly why IR papers state their
weighting scheme instead of saying "we used TF-IDF".

The practical rule: **you may not compare a cosine value against one computed with a
different weighting scheme.** Comparing rankings is fine. Comparing 0.7469 against
0.8354 is meaningless — you saw them come off the same four documents.

---

## Part 4 — Real scale, and a search engine

2,000 film reviews, the same corpus as lab 6. First, vectorize the lot.

In [ ]:
from nltk.corpus import movie_reviews

ids = movie_reviews.fileids()
texts = [movie_reviews.raw(i) for i in ids]
labels = np.array([1 if i.startswith("pos") else 0 for i in ids])

print("documents:", len(texts), "—", int(labels.sum()), "positive,",
      int((1 - labels).sum()), "negative")

tfidf = TfidfVectorizer(min_df=2)     # min_df=2: drop words that appear in only one review
M = tfidf.fit_transform(texts)

print("matrix shape:", M.shape)
print("stored (non-zero) values:", M.nnz)
print("density: %.3f%% — so %.1f%% of the matrix is zeros"
      % (100 * M.nnz / (M.shape[0] * M.shape[1]),
         100 - 100 * M.nnz / (M.shape[0] * M.shape[1])))

# Verified output:
#   documents: 2000 — 1000 positive, 1000 negative
#   matrix shape: (2000, 24087)
#   stored (non-zero) values: 651270
#   density: 1.352% — so 98.6% of the matrix is zeros

**98.6% zeros.** That is the sparsity session 17 promised, at a scale you can now see:
as a dense array this would be 48 million floats; stored sparsely it is 651 thousand.

### A detail worth stopping for

`TfidfVectorizer` L2-normalises every row by default — `norm='l2'`. So every document
vector it hands you is already a **unit vector**. Session 19 framed cosine as *the dot
product of two unit vectors*, and presented that as a way to think about it.

It is not just a way to think about it. It is how the library is built.

In [ ]:
from sklearn.metrics.pairwise import linear_kernel

norms = np.linalg.norm(M[:5].toarray(), axis=1)
print("length of the first five document vectors:", np.round(norms, 10))

q = tfidf.transform(["star wars space battle"])
print("\ncosine_similarity and a plain dot product agree:",
      np.allclose(cosine_similarity(q, M), linear_kernel(q, M)))

print("\nsklearn's own docs put it like this:")
print("  'The cosine similarity between two vectors is their dot product")
print("   when l2 norm has been applied.'")

# Verified output:
#   length of the first five document vectors: [1. 1. 1. 1. 1.]
#   cosine_similarity and a plain dot product agree: True

### Now build the search engine

This is session 10's ranking, done the way a real engine does it, and it is four lines:

1. Turn the query into a vector with the **same** vectorizer (`.transform`, never
   `.fit_transform` — the idf values must come from the collection, not from the query).
2. Cosine it against every document.
3. Sort.
4. Return the top *k*.

In [ ]:
def search(query, k=3):
    """Return the k documents most similar to the query, as (score, doc_id, snippet)."""
    # YOUR CODE HERE
    # tfidf.transform([query]) -> cosine_similarity against M -> np.argsort
    # Remember argsort is ascending; you want the largest scores.
    pass

In [ ]:
for query_text in ["star wars space battle",
                   "romantic comedy wedding love",
                   "horror slasher blood"]:
    print(f"### {query_text}")
    for s, doc_id, snippet in search(query_text):
        print(f"  {s:.4f}  {doc_id}")
        print(f"          {snippet} ...")
    print()

**Try your own queries** before reading on. Two things are worth noticing.

**The scores are low.** The best match for a good query is around 0.2–0.3, nowhere near
1. Part 5 explains why, and the reason is more interesting than "the corpus is small".

**Some hits are wrong in an instructive way.** `star wars space battle` returns, in
second place, a review about *legal systems* — it earns its score on `battle` and
`wars` used as metaphors. The engine has no idea what a word means. It is matching
strings, weighted cleverly.

---

## Part 5 — Why every pair of documents looks similar

Session 18's central callout: if a word appears in **every** document then `df = N`, so
`idf = log(N/N) = 0` — *exactly* zero — and the word contributes nothing at all.

That is airtight arithmetic. Now check whether it ever actually happens.

In [ ]:
cv_all = CountVectorizer()
C = cv_all.fit_transform(texts)
df_all = np.asarray((C > 0).sum(axis=0)).ravel()
vocab_all = cv_all.get_feature_names_out()

print("vocabulary size:", len(vocab_all))
print("words appearing in ALL 2000 reviews (idf exactly 0):",
      int((df_all == len(texts)).sum()))

print("\nthe eight most widespread words:")
for i in np.argsort(-df_all)[:8]:
    print(f"  {vocab_all[i]:>6}  in {df_all[i]} of 2000 reviews")

# Verified output:
#   vocabulary size: 39659
#   words appearing in ALL 2000 reviews (idf exactly 0): 0
#   the eight most widespread words:
#      the  in 1999 of 2000 reviews
#      and  in 1998 of 2000 reviews
#       of  in 1998 of 2000 reviews
#       to  in 1997 of 2000 reviews
#       is  in 1995 of 2000 reviews
#       in  in 1994 of 2000 reviews
#       it  in 1967 of 2000 reviews
#     that  in 1963 of 2000 reviews

**Not one word.** Somewhere in the collection there is a review that never uses `the`.

So the exactly-zero case session 18 built an hour around **never fires on this data**.
`the` gets `idf = log(2000/1999)`, which is tiny but not zero, and it appears in nearly
every document. So does `and`, and `of`, and `to`. Every document is carrying a little
bit of weight on words that every other document also has.

Measure what that does.

In [ ]:
rng = np.random.default_rng(42)          # seeded, so everyone gets the same sample
sample = rng.choice(len(texts), 200, replace=False)

def mean_pairwise_cosine(**kwargs):
    Mv = TfidfVectorizer(min_df=2, **kwargs).fit_transform(texts)
    S = cosine_similarity(Mv[sample])
    off_diagonal = S[~np.eye(len(sample), dtype=bool)]
    return float(off_diagonal.mean())

print("mean cosine between two random reviews:\n")
print("  plain TF-IDF                %.4f" % mean_pairwise_cosine())
print("  + stop_words='english'      %.4f" % mean_pairwise_cosine(stop_words="english"))
print("  + sublinear_tf=True         %.4f" % mean_pairwise_cosine(sublinear_tf=True))

# Verified output:
#   plain TF-IDF                0.2197
#   + stop_words='english'      0.0304
#   + sublinear_tf=True         0.0791

**Two unrelated film reviews are 22% similar** before you do anything about it — and
removing stop words drops that to 3%.

This is worth being careful about, because it is easy to draw the wrong conclusion.
It does **not** mean idf failed. idf did exactly what session 18 said: it shrank the
common words hard. It just could not shrink them to *nothing*, because nothing here has
`df = N`, and 39,000 slightly-weighted shared function words add up.

And notice the third line. `sublinear_tf=True` is session 18's log damping —
`1 + log(count)` — which does most of the same job without any word list at all, by
stopping a word that appears 80 times in one review from counting 80 times as much as a
word that appears once.

(One detail, since Part 3 made a point of it: sklearn's `sublinear_tf` uses the
**natural** log, while session 18 used log₁₀. Same shape, different base — and because
of the `+1` in front, the two are not simply a constant factor apart, so they give
genuinely different vectors. The idea is the lecture's; the exact numbers are sklearn's.)

That is session 4 arriving from a third direction. Session 4 removed stop words from a
human-written list. Session 18 derived the same conclusion from the collection. Here
the log damping gets much of the way there with neither.

---

## Part 6 — Does TF-IDF actually help?

Nobody has asked this yet. Sessions 18 and 19 showed what TF-IDF *does*; this asks
whether it is worth doing.

There is a clean way to find out. Lab 6 built a sentiment classifier on raw counts and
got **0.824**. Hold the classifier fixed, change only the representation, and see what
happens to the accuracy. That is a controlled experiment, and it is also exactly the
baseline lab 8 will need when it compares TF-IDF against Word2Vec and GloVe.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

Xtr, Xte, ytr, yte = train_test_split(
    texts, labels, test_size=500, random_state=42, stratify=labels)

def accuracy(vectorizer):
    A = vectorizer.fit_transform(Xtr)
    return MultinomialNB().fit(A, ytr).score(vectorizer.transform(Xte), yte)

print("lab 6's baseline, raw counts :", round(accuracy(CountVectorizer()), 4))
print("TF-IDF                       :", round(accuracy(TfidfVectorizer()), 4))

# Verified output:
#   lab 6's baseline, raw counts : 0.824
#   TF-IDF                       : 0.806

TF-IDF is **worse**. 0.806 against 0.824.

Before concluding anything: lab 6 spent its last section establishing that a single
split cannot settle a difference this small. Accuracy on one split moved between 0.776
and 0.842 there depending only on the seed — a swing nearly four times the gap we are
looking at.

So do it properly. Twenty splits, both representations scored on **the same** split each
time, and count wins rather than averaging blindly.

In [ ]:
def paired_trial(make_a, make_b, n=20):
    """Score two vectorizers on the same 20 splits. Returns two arrays of accuracies."""
    # YOUR CODE HERE
    # for seed in range(n): make ONE split, then score both vectorizers on it.
    # The point of 'paired' is that both see identical training and test data.
    pass

In [ ]:
from math import comb

def sign_test_p(wins, losses):
    """Two-sided sign test: if the two were equally good, how surprising is this record?"""
    n = wins + losses
    if n == 0:
        return 1.0
    # min(1.0, ...) because doubling the one-sided tail over-counts the shared
    # central term when wins == losses — without it, 9 wins and 9 losses would
    # report p = 1.19, which is not a probability.
    return min(1.0, 2 * sum(comb(n, k) for k in range(min(wins, losses) + 1)) / 2 ** n)

def report(name, a, b):
    """a is the baseline, b is the challenger."""
    d = b - a
    wins, losses, ties = int((d > 0).sum()), int((d < 0).sum()), int((d == 0).sum())
    print(f"{name}")
    print(f"   mean difference {d.mean():+.4f}   (std of the difference {d.std():.4f})")
    print(f"   challenger wins {wins}, ties {ties}, loses {losses}"
          f"   ->  sign test p = {sign_test_p(wins, losses):.4f}")
    print()

report("TF-IDF  vs  raw counts", counts_acc, tfidf_acc)

# Verified output:
#   TF-IDF  vs  raw counts
#      mean difference -0.0050   (std of the difference 0.0094)
#      challenger wins 4, ties 2, loses 14   ->  sign test p = 0.0309

Read that carefully, because it is a **weak** result and it should be reported as one.

TF-IDF loses 14 of the 18 splits that were not ties, which would be a fairly unlikely
record if the two were equally good (p ≈ 0.03). But the *size* of the effect is 0.005,
and the split-to-split noise is 0.009 — nearly twice as big. So: TF-IDF probably hurts
a little here, and anyone claiming a specific number for how much is over-reading their
data.

One honest caveat on that p-value, since this section is about statistical care. The 20
splits all resample the **same** 2,000 documents, so they are not 20 independent
experiments — they overlap heavily, which makes the sign test's assumption of
independent trials optimistic. Treat p ≈ 0.03 as "worth a second look", not as a result
you would put in a paper. The win/loss record is the more honest summary, and it is why
this notebook prints it.

### Now change one thing

Session 18's tf was `1 + log₁₀(count)`. sklearn's default tf is the **raw count** — it
does not apply the lecture's log damping unless you ask. Ask.

In [ ]:
sublinear = lambda: TfidfVectorizer(sublinear_tf=True)

# One more paired run — about 20 seconds. We only need its second return value:
# the first re-measures plain TF-IDF, which we already have in tfidf_acc from
# the identical seeds 0..19, so the comparisons below stay properly paired.
print("Running one more paired trial — about 20 seconds.\n")
_, sub_acc = paired_trial(TfidfVectorizer, sublinear)

report("sublinear_tf=True  vs  plain TF-IDF", tfidf_acc, sub_acc)
report("sublinear_tf=True  vs  raw counts (lab 6's baseline)", counts_acc, sub_acc)

print("means:  raw counts %.4f | plain TF-IDF %.4f | sublinear TF-IDF %.4f"
      % (counts_acc.mean(), tfidf_acc.mean(), sub_acc.mean()))

# Verified output:
#   sublinear_tf=True  vs  plain TF-IDF
#      mean difference +0.0188   (std of the difference 0.0088)
#      challenger wins 19, ties 0, loses 1   ->  sign test p = 0.0000
#
#   sublinear_tf=True  vs  raw counts (lab 6's baseline)
#      mean difference +0.0138   (std of the difference 0.0088)
#      challenger wins 19, ties 1, loses 0   ->  sign test p = 0.0000
#
#   means:  raw counts 0.8094 | plain TF-IDF 0.8044 | sublinear TF-IDF 0.8232

### This one is not weak

**19 wins out of 20.** Mean improvement +0.019, which is now *twice* the split-to-split
noise rather than half of it, and it beats lab 6's raw-count baseline 19–1–0.

Compare the two results honestly:

| | effect | noise | record | verdict |
|---|---|---|---|---|
| TF-IDF vs counts | −0.005 | 0.009 | 4 / 2 / 14 | probably slightly worse, small |
| **sublinear TF-IDF vs plain** | **+0.019** | 0.009 | **19 / 0 / 1** | **real** |
| **sublinear TF-IDF vs counts** | **+0.014** | 0.009 | **19 / 1 / 0** | **real** |

And notice what `sublinear_tf=True` actually is. It is one line:

```
tf = 1 + log(count)
```

which is **session 18's log damping** — the tf the lecture used throughout, and the one
sklearn leaves switched off. (In natural log rather than base 10, as Part 5 noted. The
base changes the numbers; it does not change which idea is being tested.)

Session 19 showed, on four toy vectors, that log damping was what pulled the Shakespeare
cosines apart — without it every play sat crushed up against 1 and the genre signal
survived only in the fourth decimal place. That was a lecture illustration on 16 numbers.
Here the same idea is worth about two accuracy points on 2,000 real documents and a task
the lecture never mentioned — and it wins 19 times out of 20.

**The weighting scheme is a real choice with measurable consequences.** That is the
sentence session 19 ended on, and this is the first time in the course you have
measured it.

---

## Part 7 — What cosine cannot see

One last measurement. If two reviews are near neighbours by cosine, do they agree about
the film?

In [ ]:
S_all = cosine_similarity(M)
np.fill_diagonal(S_all, -1)              # a document is not its own neighbour
nearest = S_all.argmax(axis=1)

agree = float((labels == labels[nearest]).mean())
print("nearest neighbour has the same sentiment label: %.1f%% of 2000 documents" % (agree * 100))
print("chance would be                                 50.0%")
print("lab 6's classifier managed                      ~82%")
print("\nmean cosine to the nearest neighbour: %.3f" % S_all.max(axis=1).mean())

# Verified output:
#   nearest neighbour has the same sentiment label: 67.5% of 2000 documents
#   chance would be                                 50.0%
#   lab 6's classifier managed                      ~82%
#   mean cosine to the nearest neighbour: 0.482

**67.5%.** Better than chance, well short of a classifier.

Which makes sense once you say what cosine is measuring: two reviews of the same film
share the title, the actors, the director and the plot, and those words dominate the
vector. Whether the reviewer *liked* it turns on a handful of words that the shared
vocabulary drowns out. Cosine on TF-IDF finds **topic**, and sentiment is only weakly
visible through it.

That is session 17's negation problem arriving from a new direction. There, `the film
was good, not bad` and `the film was bad, not good` produced identical bags of words.
Here, two reviews that disagree completely come out as near neighbours. Both are the
same limitation: **a bag of words records which words are present, and nothing about
what they are doing.**

### One more thing that matrix will tell you

Session 19 opened on a document and the same document printed twice, which have a cosine
of exactly 1. That was a thought experiment.

Look for it in the real corpus.

In [ ]:
close = np.argwhere(S_all > 0.999)
seen, dupes = set(), []
for i, j in close:
    if (j, i) in seen:
        continue
    seen.add((i, j))
    dupes.append((ids[i], ids[j], float(S_all[i, j])))

print("pairs of DIFFERENT reviews with cosine > 0.999:", len(dupes), "\n")
for a, b, s in dupes:
    print(f"  {s:.4f}   {a}   ~   {b}")

a, b = dupes[0][0], dupes[0][1]
ta, tb = movie_reviews.raw(a), movie_reviews.raw(b)
print(f"\nare they byte-identical? {ta == tb}   (lengths {len(ta)} and {len(tb)})")
print("first 90 characters of each:")
print("  ", ta[:90].replace("\n", " "))
print("  ", tb[:90].replace("\n", " "))

# Verified output:
#   pairs of DIFFERENT reviews with cosine > 0.999: 4
#
#     1.0000   neg/cv274_26379.txt   ~   neg/cv302_26481.txt
#     1.0000   neg/cv412_25254.txt   ~   neg/cv656_25395.txt
#     1.0000   pos/cv115_25396.txt   ~   pos/cv274_25253.txt
#     0.9997   pos/cv383_13116.txt   ~   pos/cv986_13527.txt
#
#   are they byte-identical? False   (lengths 4385 and 4394)
#   first 90 characters of each:
#      the tagline for this film is : " some houses are just born bad " .
#      the tagline for this film is : " some houses are just born bad " .

**There are four duplicate pairs in this corpus, and nobody told you.**

`movie_reviews` is a standard benchmark dataset, used in hundreds of papers, and it
contains the same review filed twice under different names — not byte-identical (4385
characters against 4394, so somebody edited a word or two) but identical in bag-of-words
terms, which is why the cosine is 1.0000 rather than 0.99.

Two things follow, and both are practical.

**Cosine ≈ 1 is how you find duplicates.** This is one of the most common industrial
uses of everything in this lab — de-duplicating a crawl, spotting plagiarism, catching
a scraper that fetched the same page twice. You did not write any special code for it;
you sorted a similarity matrix.

**And it is a small flaw in lab 6's experiment.** One or two of these four pairs land
with one copy in training and the other in test on any given split, which means a couple
of test documents have a near-identical twin the classifier has already seen. That is
**data leakage**, and it inflates the accuracy.

Be precise about the size of it before anyone panics: that is 1–2 documents out of a
500-document test set, about 0.3%, far too small to explain a result of 0.82, and it
does not change any conclusion in lab 6 or in Part 6 above. But the *principle* matters
— on a smaller corpus, or one with more duplication, exactly this can produce a model
that looks excellent and is partly just remembering. Checking for near-duplicates before
you split is a habit worth having, and you now know how.

### Where this goes

You now have two baselines on this corpus, both with the same classifier and the same
20-split protocol, both measured in this notebook:

- raw counts (lab 6's representation) — **0.809**
- sublinear TF-IDF (this lab's) — **0.823**

Lab 8 asks whether **Word2Vec** and **GloVe** vectors beat those, on this same task.
Those representations are built to carry meaning rather than to count strings, so the
numbers above are the thing they have to beat. Sessions 21 and 22 build them; keep this
notebook.